# NiyamTrace-X — Final External Experiment Completion

Publication-run notebook for completing **external validation** and downloading every result artifact.

**Required gate:** BFCL-v4 + AgentDojo + τ³, across three independent model families.

**Optional:** AgentDyn, MLCL official release, MCP-SafetyBench.

Rules: a zero exit code is not evidence unless native benchmark outputs contain scored cases; successful slices are checkpointed and reused; all raw outputs/logs/environment metadata are archived; secrets are never written to disk; `NTX_SELFTEST=1` tests parsers/exports without API spend and is always marked non-paper evidence.

In [ ]:
# CELL 1 — CONFIGURATION / WORKSPACE
from pathlib import Path
from getpass import getpass
from datetime import datetime,timezone
import os,sys,json,re,time,random,hashlib,zipfile,shutil,subprocess,platform
import numpy as np, pandas as pd, matplotlib.pyplot as plt
SEED=42; random.seed(SEED); np.random.seed(SEED)
SELFTEST=os.getenv('NTX_SELFTEST','0')=='1'
MODE=os.getenv('NTX_RUN_MODE','QUICK').upper(); assert MODE in {'QUICK','STANDARD','FULL'}
BASE=Path(os.getenv('NTX_BASE_DIR','/content/NTX_FINAL_EXTERNAL' if Path('/content').exists() else str(Path.cwd()/'NTX_FINAL_EXTERNAL')))
WORK=BASE/'work'; RESULTS=BASE/'results'; RAW=BASE/'raw'; LOGS=BASE/'logs'; ENV=BASE/'environment'; PAPER=BASE/'paper_artifacts'; ARCH=BASE/'archives'
for p in [BASE,WORK,RESULTS,RAW,LOGS,ENV,PAPER,ARCH]: p.mkdir(parents=True,exist_ok=True)
CFG={
'QUICK':dict(bfcl_limit=5,dojo_suites=['banking'],dojo_users=['user_task_0'],dojo_injections=['injection_task_0'],tau_domains=['airline','retail','telecom'],tau_tasks=1,tau_steps=18,agent_tokens=1024,user_tokens=512,bootstrap=1000),
'STANDARD':dict(bfcl_limit=50,dojo_suites=['banking','workspace','travel','slack'],dojo_users=['user_task_0','user_task_5','user_task_10'],dojo_injections=['injection_task_0','injection_task_1'],tau_domains=['airline','retail','telecom'],tau_tasks=20,tau_steps=40,agent_tokens=1536,user_tokens=768,bootstrap=5000),
'FULL':dict(bfcl_limit=None,dojo_suites=['banking','workspace','travel','slack'],dojo_users=None,dojo_injections=None,tau_domains=['airline','retail','telecom'],tau_tasks=None,tau_steps=80,agent_tokens=2048,user_tokens=1024,bootstrap=10000)}[MODE]
ENABLE_AGENTDYN=os.getenv('NTX_ENABLE_AGENTDYN','0')=='1'; ENABLE_MCP=os.getenv('NTX_ENABLE_MCP','0')=='1'; MLCL_OFFICIAL_PATH=os.getenv('MLCL_OFFICIAL_PATH','').strip(); DRIVE_BACKUP=os.getenv('NTX_DRIVE_BACKUP_DIR','').strip()
print('MODE',MODE,'SELFTEST',SELFTEST,'BASE',BASE); print(json.dumps(CFG,indent=2))

In [ ]:
# CELL 2 — HELPERS / CHECKPOINTS
CHECKPOINT=RESULTS/'checkpoint_state.json'
def now(): return datetime.now(timezone.utc).isoformat()
def run(cmd,cwd=None,env=None,timeout=None,check=False):
    p=subprocess.run([str(x) for x in cmd],cwd=str(cwd) if cwd else None,env=env,capture_output=True,text=True,errors='replace',timeout=timeout)
    if check and p.returncode: raise RuntimeError((p.stderr or p.stdout)[-5000:])
    return p
def logproc(name,p,cmd=None):
    (LOGS/name).write_text(('COMMAND\n'+' '.join(map(str,cmd))+'\n\n' if cmd else '')+'STDOUT\n'+(p.stdout or '')+'\n\nSTDERR\n'+(p.stderr or ''),errors='ignore')
def sha(p):
    h=hashlib.sha256(); f=open(p,'rb')
    for c in iter(lambda:f.read(1024*1024),b''): h.update(c)
    f.close(); return h.hexdigest()
def jd(p,o): Path(p).write_text(json.dumps(o,indent=2,default=str))
def state_load():
    try:return json.loads(CHECKPOINT.read_text()) if CHECKPOINT.exists() else {}
    except:return {}
STATE=state_load()
def mark(k,status,**kw): STATE[k]=dict(status=status,updated_at=now(),**kw); jd(CHECKPOINT,STATE)
def done(k): return STATE.get(k,{}).get('status')=='SUPPORTED'
def ensure_uv():
    if not shutil.which('uv'):
        p=run([sys.executable,'-m','pip','install','-q','-U','uv']); logproc('install_uv.log',p)
        if p.returncode: raise RuntimeError('uv install failed')
def clone(url,dst):
    dst=Path(dst)
    if not dst.exists():
        p=run(['git','clone','--depth','1',url,dst]); logproc('clone_'+dst.name+'.log',p)
        if p.returncode: raise RuntimeError('clone failed '+url)
    return run(['git','-C',dst,'rev-parse','HEAD'],check=True).stdout.strip()
def cp_tree(src,dst):
    src,dst=Path(src),Path(dst)
    if not src.exists(): return False
    shutil.rmtree(dst,ignore_errors=True); shutil.copytree(src,dst); return True
def zip_tree(src,dst):
    src,dst=Path(src),Path(dst); dst.unlink(missing_ok=True)
    with zipfile.ZipFile(dst,'w',zipfile.ZIP_DEFLATED,allowZip64=True) as z:
        for p in sorted(src.rglob('*')):
            if p.is_file(): z.write(p,str(p.relative_to(src)))
    return dst
print('checkpoint entries',len(STATE))

## Provider preflight
The notebook uses one OpenRouter key and live model-catalog discovery. QUICK/STANDARD prefer cheaper fallbacks; FULL prefers the configured primary models.

In [ ]:
# CELL 3 — MODEL CATALOG / BUDGET / TOOL-CALL PROBE
import urllib.request
API='https://openrouter.ai/api/v1'
KEY='SELFTEST' if SELFTEST else (os.getenv('OPENROUTER_API_KEY','').strip() or getpass('OpenRouter API key (hidden): ').strip())
if not KEY: raise RuntimeError('OPENROUTER_API_KEY required')
def http(method,url,body=None,timeout=60):
    req=urllib.request.Request(url,data=json.dumps(body).encode() if body is not None else None,headers={'Authorization':'Bearer '+KEY,'Content-Type':'application/json','HTTP-Referer':'https://openai.com/','X-Title':'NiyamTrace-X validation'},method=method)
    with urllib.request.urlopen(req,timeout=timeout) as r:return r.status,json.loads(r.read().decode())
MATRIX=json.loads(os.getenv('NTX_MODEL_MATRIX_JSON','null') or 'null') or [
 dict(label='qwen',family='Qwen',primary='qwen/qwen3-coder:exacto',fallbacks=['qwen/qwen3-30b-a3b-instruct-2507']),
 dict(label='gpt-oss',family='GPT-OSS',primary='openai/gpt-oss-120b:exacto',fallbacks=['openai/gpt-oss-20b']),
 dict(label='glm',family='GLM',primary='z-ai/glm-4.6:exacto',fallbacks=['z-ai/glm-4.5-air'])]
if SELFTEST:
    catalog={'data':[{'id':'qwen/qwen3-30b-a3b-instruct-2507','pricing':{'prompt':'0.000000048','completion':'0.000000193'}},{'id':'openai/gpt-oss-20b','pricing':{'prompt':'0.0000001','completion':'0.0000005'}},{'id':'z-ai/glm-4.5-air','pricing':{'prompt':'0.00000013','completion':'0.00000085'}}]}; keyinfo={'selftest':True,'limit_remaining':999}
else:
    _,catalog=http('GET',API+'/models')
    try: _,keyinfo=http('GET',API+'/key')
    except Exception as e:keyinfo={'status':'UNAVAILABLE','error':repr(e)}
jd(ENV/'openrouter_catalog.json',catalog); jd(ENV/'openrouter_key_status_redacted.json',{k:v for k,v in keyinfo.items() if 'key' not in k.lower() and 'token' not in k.lower()})
ids={x.get('id') for x in catalog.get('data',[]) if isinstance(x,dict)}
def price(mid):
    x=next((x for x in catalog.get('data',[]) if x.get('id')==mid),{}); pr=x.get('pricing') or {}
    try:return float(pr.get('prompt','inf'))+float(pr.get('completion','inf'))
    except:return float('inf')
PROFILE=os.getenv('NTX_MODEL_PROFILE','PRIMARY' if MODE=='FULL' else 'BUDGET').upper(); models=[]
for s in MATRIX:
    av=[x for x in [s['primary']]+s.get('fallbacks',[]) if x in ids]
    chosen=(s['primary'] if PROFILE=='PRIMARY' and s['primary'] in av else (min(av,key=price) if av else None))
    models.append({**s,'model':chosen,'selection':'SELECTED' if chosen else 'UNAVAILABLE'})
# Chat+tool probe.
rows=[]; tool=[{'type':'function','function':{'name':'weather','description':'Weather','parameters':{'type':'object','properties':{'city':{'type':'string'}},'required':['city']}}}]
for m in models:
    if SELFTEST: rows.append(dict(label=m['label'],family=m['family'],model=m['model'],chat_ok=True,tool_ok=True,status='SELFTEST_ONLY')); continue
    if not m['model']: rows.append(dict(label=m['label'],family=m['family'],model=None,chat_ok=False,tool_ok=False,status='NO_MODEL')); continue
    row=dict(label=m['label'],family=m['family'],model=m['model'])
    try:
        _,o=http('POST',API+'/chat/completions',{'model':m['model'],'messages':[{'role':'user','content':'Reply exactly OK'}],'temperature':0,'max_tokens':16}); row['chat_ok']=bool(o.get('choices'))
    except Exception as e: row['chat_ok']=False; row['chat_error']=repr(e)
    try:
        _,o=http('POST',API+'/chat/completions',{'model':m['model'],'messages':[{'role':'user','content':'Use the weather tool for Chennai'}],'tools':tool,'tool_choice':'auto','temperature':0,'max_tokens':96}); msg=(o.get('choices') or [{}])[0].get('message') or {}; row['tool_ok']=bool(msg.get('tool_calls'))
    except Exception as e: row['tool_ok']=False; row['tool_error']=repr(e)
    row['status']='OK' if row.get('chat_ok') and row.get('tool_ok') else 'FAILED'; rows.append(row)
preflight=pd.DataFrame(rows); preflight.to_csv(RESULTS/'00_provider_preflight.csv',index=False); display(preflight)
WORKING=[m for m in models if len(preflight[(preflight.label==m['label']) & preflight.chat_ok.astype(bool) & preflight.tool_ok.astype(bool)])]
if not SELFTEST and not WORKING: raise RuntimeError('No model passed chat+tool preflight')
if MODE=='FULL' and not SELFTEST and len({m['family'] for m in WORKING})<3: raise RuntimeError('FULL requires three working families')
jd(ENV/'selected_models.json',[{k:v for k,v in m.items() if k not in {'api_key'}} for m in WORKING])

In [ ]:
# CELL 4 — HOST REPRODUCIBILITY SNAPSHOT
jd(ENV/'system_info.json',dict(timestamp=now(),python=sys.version,platform=platform.platform(),mode=MODE,selftest=SELFTEST,config=CFG,profile=PROFILE))
(ENV/'host_pip_freeze.txt').write_text(run([sys.executable,'-m','pip','freeze']).stdout or '')

# Required 1 — BFCL-v4
Clean Python 3.11 environment using `evalscope[bfcl]`. Native reports must contain numeric scores and evaluated cases.

In [ ]:
# CELL 5 — BFCL ENVIRONMENT
BFCL_ENV=WORK/'bfcl_env'
if SELFTEST: BFCL_PY=Path(sys.executable)
else:
    ensure_uv(); run(['uv','python','install','3.11'])
    if BFCL_ENV.exists() and os.getenv('NTX_REBUILD_BFCL','0')=='1': shutil.rmtree(BFCL_ENV)
    if not BFCL_ENV.exists():
        p=run(['uv','venv',BFCL_ENV,'--python','3.11']); logproc('bfcl_venv.log',p)
        if p.returncode: raise RuntimeError('BFCL venv failed')
    BFCL_PY=BFCL_ENV/'bin'/'python'; BFCL_PY=BFCL_PY if BFCL_PY.exists() else BFCL_ENV/'Scripts'/'python.exe'
    p=run(['uv','pip','install','--python',BFCL_PY,'-U','evalscope[bfcl]']); logproc('bfcl_install.log',p)
    if p.returncode: raise RuntimeError('BFCL install failed; inspect log')
    v=run([BFCL_PY,'-c',"import importlib.metadata as m; from evalscope import run_task; from evalscope.config import TaskConfig; print(m.version('evalscope')); print(m.version('bfcl-eval')); print('BFCL_READY')"]); logproc('bfcl_verify.log',v)
    if v.returncode or 'BFCL_READY' not in v.stdout: raise RuntimeError('BFCL verify failed')
    (ENV/'bfcl_pip_freeze.txt').write_text(run([BFCL_PY,'-m','pip','freeze']).stdout or '')
print('BFCL python',BFCL_PY)

In [ ]:
# CELL 6 — BFCL RUN + PARSE
def parse_bfcl(label,root):
    root=Path(root); rows=[]; n=0
    for p in root.rglob('*'):
        if not p.is_file() or p.suffix.lower() not in {'.csv','.json','.jsonl'}: continue
        rel=str(p.relative_to(root)); low=rel.lower()
        try:
            if p.suffix.lower()=='.csv':
                d=pd.read_csv(p); n=max(n,len(d) if any(x in low for x in ['report','review','prediction']) else 0)
                for c in d.columns:
                    if any(x in str(c).lower() for x in ['accuracy','overall_score','score']):
                        for v in pd.to_numeric(d[c],errors='coerce').dropna(): rows.append(dict(benchmark='BFCL-v4',model=label,slice=rel,metric=str(c),score=float(v),source_file=rel))
            else:
                texts=p.read_text(errors='ignore').splitlines() if p.suffix.lower()=='.jsonl' else [p.read_text(errors='ignore')]
                for t in texts:
                    try:o=json.loads(t)
                    except:continue
                    stack=[('',o)]
                    while stack:
                        path,x=stack.pop()
                        if isinstance(x,dict):
                            for ck in ['total_count','num_samples','n_samples','evaluated_count']:
                                if isinstance(x.get(ck),(int,float)) and x[ck]>0:n=max(n,int(x[ck]))
                            for k,v in x.items():
                                q=f'{path}.{k}' if path else str(k)
                                if isinstance(v,(dict,list)):stack.append((q,v))
                                elif isinstance(v,(int,float)) and any(a in str(k).lower() for a in ['accuracy','overall_score','score']): rows.append(dict(benchmark='BFCL-v4',model=label,slice=rel,metric=q,score=float(v),source_file=rel))
                        elif isinstance(x,list):
                            if x and any(a in low for a in ['review','prediction']):n=max(n,len(x))
                            for i,v in enumerate(x):stack.append((f'{path}[{i}]',v))
        except: pass
    return pd.DataFrame(rows).drop_duplicates() if rows else pd.DataFrame(columns=['benchmark','model','slice','metric','score','source_file']),n
parts=[]; st=[]
for m in WORKING:
    label=m['label']; root=RAW/'bfcl'/label/MODE.lower(); root.mkdir(parents=True,exist_ok=True); key=f'bfcl::{label}::{MODE}'
    if SELFTEST:
        (root/'report.json').write_text(json.dumps({'accuracy':0.75,'total_count':4,'evidence_tag':'NON_PAPER_SELFTEST'})); p=None
    elif not done(key):
        script=root/'run_bfcl.py'; script.write_text(f"""from evalscope import run_task\nfrom evalscope.config import TaskConfig\ncfg=TaskConfig(model={m['model']!r},api_url={API!r},api_key={KEY!r},eval_type='openai_api',datasets=['bfcl_v4'],work_dir={str(root)!r},limit={CFG['bfcl_limit']!r},seed={SEED},generation_config={{'temperature':0.0,'max_tokens':1024,'retries':1,'retry_interval':2,'timeout':120}},dataset_args={{'bfcl_v4':{{'extra_params':{{'is_fc_model':True}}}}}})\nrun_task(task_cfg=cfg)\n""")
        p=run([BFCL_PY,script],cwd=root,timeout=None); logproc(f'bfcl_{label}_{MODE}.log',p,[BFCL_PY,script])
    else:p=None
    d,n=parse_bfcl(label,root); parts.append(d); status=('SELFTEST_ONLY' if SELFTEST else ('SUPPORTED' if n>0 and len(d)>0 else 'INFRA_FAILURE')); rc=0 if p is None else p.returncode
    if not SELFTEST:mark(key,status,returncode=rc,evaluated_count=n,numeric_metrics=len(d))
    st.append(dict(model=label,family=m['family'],status=status,evaluated_count=n,numeric_metrics=len(d),returncode=rc))
bfcl=pd.concat(parts,ignore_index=True) if parts else pd.DataFrame(); bfcl_status=pd.DataFrame(st); bfcl.to_csv(RESULTS/'10_bfcl_metrics.csv',index=False); bfcl_status.to_csv(RESULTS/'10_bfcl_status.csv',index=False); display(bfcl_status); display(bfcl.head(30))

# Required 2 — AgentDojo
Uses `openai-compatible`, `--model-id`, unique logdirs and `--force-rerun`. JSON result files are parsed directly for benchmark-native `utility` and `security`.

In [ ]:
# CELL 7 — AGENTDOJO ENVIRONMENT
DOJO=WORK/'agentdojo'
if SELFTEST: DOJO_COMMIT='SELFTEST'
else:
    DOJO_COMMIT=clone('https://github.com/ethz-spylab/agentdojo.git',DOJO); ensure_uv(); p=run(['uv','sync'],cwd=DOJO); logproc('dojo_sync.log',p)
    if p.returncode:raise RuntimeError('AgentDojo sync failed')
    hp=run(['uv','run','python','-m','agentdojo.scripts.benchmark','--help'],cwd=DOJO); logproc('dojo_help.log',hp); txt=(hp.stdout or '')+(hp.stderr or '')
    if not all(x in txt for x in ['openai-compatible','--model-id','--force-rerun']):raise RuntimeError('AgentDojo required interface unavailable')
    (ENV/'agentdojo_pip_freeze.txt').write_text(run(['uv','pip','freeze'],cwd=DOJO).stdout or ''); jd(ENV/'agentdojo_version.json',{'commit':DOJO_COMMIT})
print('AgentDojo',DOJO_COMMIT)

In [ ]:
# CELL 8 — AGENTDOJO RUN + PARSE
def parse_dojo(label,suite,root):
    rows=[]; root=Path(root)
    for p in root.rglob('*.json'):
        try:o=json.loads(p.read_text())
        except:continue
        if not isinstance(o,dict) or ('utility' not in o and 'security' not in o):continue
        u,s=o.get('utility'),o.get('security'); rows.append(dict(benchmark='AgentDojo',model=label,suite=suite,user_task_id=o.get('user_task_id'),injection_task_id=o.get('injection_task_id'),utility=int(u) if isinstance(u,bool) else np.nan,security=int(s) if isinstance(s,bool) else np.nan,error=o.get('error'),source_file=str(p.relative_to(root))))
    return pd.DataFrame(rows)
parts=[]; st=[]
for m in WORKING:
  for suite in CFG['dojo_suites']:
    label=m['label']; root=RAW/'agentdojo'/label/suite/MODE.lower(); root.mkdir(parents=True,exist_ok=True); key=f'dojo::{label}::{suite}::{MODE}'
    if SELFTEST:
        q=root/'openai-compatible'/suite/'user_task_0'/'important_instructions';q.mkdir(parents=True,exist_ok=True);(q/'injection_task_0.json').write_text(json.dumps({'user_task_id':'user_task_0','injection_task_id':'injection_task_0','utility':True,'security':True,'error':None,'evidence_tag':'NON_PAPER_SELFTEST'}));p=None
    elif not done(key):
        env=os.environ.copy();env['OPENAI_COMPATIBLE_BASE_URL']=API;env['OPENAI_COMPATIBLE_API_KEY']=KEY
        cmd=['uv','run','python','-m','agentdojo.scripts.benchmark','--model','openai-compatible','--model-id',m['model'],'-s',suite,'--attack','important_instructions','--logdir',str(root),'--force-rerun','--max-workers','1']
        if CFG['dojo_users'] is not None:
            for x in CFG['dojo_users']:cmd+=['-ut',x]
        if CFG['dojo_injections'] is not None:
            for x in CFG['dojo_injections']:cmd+=['-it',x]
        p=run(cmd,cwd=DOJO,env=env,timeout=None);logproc(f'dojo_{label}_{suite}_{MODE}.log',p,cmd)
    else:p=None
    d=parse_dojo(label,suite,root);rc=0 if p is None else p.returncode
    if len(d):d['family']=m['family'];parts.append(d)
    valid=int(((d.utility.notna())|(d.security.notna())).sum()) if len(d) else 0;errs=int(d.error.notna().sum()) if len(d) else 0;status='SELFTEST_ONLY' if SELFTEST else ('SUPPORTED' if valid>0 and errs==0 else ('PARTIAL' if valid>0 else 'INFRA_FAILURE'))
    if not SELFTEST:mark(key,status,returncode=rc,n_scored=valid,errors=errs)
    st.append(dict(model=label,family=m['family'],suite=suite,status=status,n_scored=valid,errors=errs,returncode=rc))
dojo=pd.concat(parts,ignore_index=True) if parts else pd.DataFrame();dojo_status=pd.DataFrame(st);dojo.to_csv(RESULTS/'11_dojo_cases.csv',index=False);dojo_status.to_csv(RESULTS/'11_dojo_status.csv',index=False)
if len(dojo):dojo_summary=dojo.groupby(['model','family','suite'],dropna=False).agg(n=('source_file','count'),utility=('utility','mean'),security=('security','mean'),errors=('error',lambda x:int(x.notna().sum()))).reset_index()
else:dojo_summary=pd.DataFrame(columns=['model','family','suite','n','utility','security','errors'])
dojo_summary.to_csv(RESULTS/'11_dojo_summary.csv',index=False);display(dojo_status);display(dojo_summary)

# Required 3 — τ³ / tau2-bench
Dependencies are verified inside tau2's own `.venv`. A fixed user simulator is used across agent models. Token/step/retry caps prevent the earlier 402-budget blowups. Code 0 with zero scored simulations is an infrastructure failure.

In [ ]:
# CELL 9 — TAU ENVIRONMENT
TAU=WORK/'tau2-bench'
if SELFTEST:TAU_COMMIT='SELFTEST';TAUPY=Path(sys.executable)
else:
    TAU_COMMIT=clone('https://github.com/sierra-research/tau2-bench.git',TAU);ensure_uv();p=run(['uv','sync'],cwd=TAU);logproc('tau_sync.log',p)
    if p.returncode:raise RuntimeError('tau sync failed')
    TAUPY=TAU/'.venv'/'bin'/'python';
    if not TAUPY.exists():raise RuntimeError('tau .venv missing')
    p=run(['uv','pip','install','--python',TAUPY,'websockets','soundfile'],cwd=TAU);logproc('tau_deps.log',p);v=run([TAUPY,'-c',"import websockets,soundfile;print('TAU_DEPS_OK')"],cwd=TAU);logproc('tau_verify.log',v)
    if v.returncode or 'TAU_DEPS_OK' not in v.stdout:raise RuntimeError('tau deps failed')
    (ENV/'tau_pip_freeze.txt').write_text(run(['uv','pip','freeze'],cwd=TAU).stdout or '');jd(ENV/'tau_version.json',{'commit':TAU_COMMIT})
print('tau2',TAU_COMMIT)

In [ ]:
# CELL 10 — TAU RUN + STRICT PARSER
def parse_tau(p):
    try:o=json.loads(Path(p).read_text())
    except:return pd.DataFrame()
    sims=[]
    if isinstance(o,dict):
        for k in ['simulations','results','trajectories']:
            if isinstance(o.get(k),list):sims=o[k];break
    elif isinstance(o,list):sims=o
    rows=[]
    for i,s in enumerate(sims):
        if not isinstance(s,dict):continue
        ri=s.get('reward_info'); reward=float(ri['reward']) if isinstance(ri,dict) and isinstance(ri.get('reward'),(int,float,bool)) else (float(s['reward']) if isinstance(s.get('reward'),(int,float,bool)) else np.nan)
        err=s.get('error');rows.append(dict(trajectory_index=i,task_id=s.get('task_id'),trial=s.get('trial'),reward=reward,error=err))
    return pd.DataFrame(rows)
FIXED_USER=next((x for x in ['qwen/qwen3-30b-a3b-instruct-2507','qwen/qwen3-coder:exacto'] if x in ids),WORKING[0]['model'] if WORKING else None)
parts=[];st=[]
for m in WORKING:
  for domain in CFG['tau_domains']:
    label=m['label'];key=f'tau::{label}::{domain}::{MODE}';run_name=f'ntx_final_{label}_{domain}_{MODE.lower()}';live=TAU/'data'/'simulations'/run_name;root=RAW/'tau'/label/domain/MODE.lower();p=None
    if SELFTEST:
        root.mkdir(parents=True,exist_ok=True);(root/'results.json').write_text(json.dumps({'simulations':[{'task_id':'0','trial':0,'reward_info':{'reward':1.0},'error':None},{'task_id':'1','trial':0,'reward_info':{'reward':0.0},'error':None}],'evidence_tag':'NON_PAPER_SELFTEST'}))
    elif not done(key):
        env=os.environ.copy();env['OPENROUTER_API_KEY']=KEY;cmd=['uv','run','tau2','run','--domain',domain,'--agent-llm','openrouter/'+m['model'],'--user-llm','openrouter/'+FIXED_USER,'--agent-llm-args',json.dumps({'temperature':0.0,'max_tokens':CFG['agent_tokens']}),'--user-llm-args',json.dumps({'temperature':0.0,'max_tokens':CFG['user_tokens']}),'--num-trials','1','--task-split-name','base','--max-steps',str(CFG['tau_steps']),'--max-errors','3','--max-concurrency','1','--max-retries','1','--retry-delay','2','--seed',str(SEED),'--save-to',run_name,'--auto-resume','--verbose-logs','--llm-log-mode','all']
        if CFG['tau_tasks'] is not None:cmd+=['--num-tasks',str(CFG['tau_tasks'])]
        p=run(cmd,cwd=TAU,env=env,timeout=None);logproc(f'tau_{label}_{domain}_{MODE}.log',p,cmd)
        if live.exists():cp_tree(live,root)
    d=parse_tau(root/'results.json') if (root/'results.json').exists() else pd.DataFrame();rc=0 if p is None else p.returncode
    if len(d):d['benchmark']='tau3';d['model']=label;d['family']=m['family'];d['domain']=domain;parts.append(d)
    n=int(d.reward.notna().sum()) if len(d) else 0;errs=int(d.error.notna().sum()) if len(d) and 'error' in d else 0;status='SELFTEST_ONLY' if SELFTEST else ('SUPPORTED' if n>0 and errs==0 else ('PARTIAL' if n>0 else 'INFRA_FAILURE'))
    if not SELFTEST:mark(key,status,returncode=rc,n_evaluated=n,infra_errors=errs,run_name=run_name,user_simulator=FIXED_USER)
    st.append(dict(model=label,family=m['family'],domain=domain,status=status,n_evaluated=n,infra_errors=errs,returncode=rc,user_simulator=FIXED_USER))
tau=pd.concat(parts,ignore_index=True) if parts else pd.DataFrame();tau_status=pd.DataFrame(st);tau.to_csv(RESULTS/'14_tau_cases.csv',index=False);tau_status.to_csv(RESULTS/'14_tau_status.csv',index=False)
if len(tau):tau_summary=tau.groupby(['model','family','domain'],dropna=False).agg(n=('reward','count'),mean_reward=('reward','mean'),infra_errors=('error',lambda x:int(x.notna().sum()))).reset_index()
else:tau_summary=pd.DataFrame(columns=['model','family','domain','n','mean_reward','infra_errors'])
tau_summary.to_csv(RESULTS/'14_tau_summary.csv',index=False);display(tau_status);display(tau_summary)

# Optional extensions
AgentDyn runs only when a compatible provider is exposed; MLCL requires an official source; MCP remains opt-in because it may perform real external actions.

In [ ]:
# CELL 11 — OPTIONAL STATUS GATES
rows=[]
if ENABLE_AGENTDYN and not SELFTEST:
    AD=WORK/'AgentDyn'
    try:
        c=clone('https://github.com/SaFo-Lab/AgentDyn.git',AD);p=run([sys.executable,'-m','pip','install','-q','-e',AD]);logproc('agentdyn_install.log',p);h=run([sys.executable,'-m','agentdojo.scripts.benchmark','--help'],cwd=AD);logproc('agentdyn_help.log',h);compat='openai-compatible' in ((h.stdout or '')+(h.stderr or ''));rows.append(dict(benchmark='AgentDyn',status='READY' if compat else 'UNSUPPORTED_PROVIDER',commit=c))
    except Exception as e:rows.append(dict(benchmark='AgentDyn',status='INFRA_FAILURE',error=repr(e)))
else:rows.append(dict(benchmark='AgentDyn',status='SELFTEST_SKIPPED' if SELFTEST else 'DISABLED'))
rows.append(dict(benchmark='MLCL',status='OFFICIAL_SOURCE_PRESENT' if MLCL_OFFICIAL_PATH and Path(MLCL_OFFICIAL_PATH).exists() else 'MISSING_OFFICIAL_SOURCE',path=MLCL_OFFICIAL_PATH))
rows.append(dict(benchmark='MCP-SafetyBench',status='ENABLED_MANUAL_ISOLATED_ONLY' if ENABLE_MCP and not SELFTEST else ('SELFTEST_SKIPPED' if SELFTEST else 'DISABLED_FOR_SAFETY')))
optional=pd.DataFrame(rows);optional.to_csv(RESULTS/'15_optional_status.csv',index=False);display(optional)

In [ ]:
# CELL 12 — UNIFIED EVIDENCE / SUMMARIES / CIs
rows=[]
for _,r in bfcl.iterrows():rows.append(dict(benchmark='BFCL-v4',model=r.model,family=next((m['family'] for m in WORKING if m['label']==r.model),''),slice=r['slice'],metric=r.metric,score=r.score,evidence_state='SELFTEST_ONLY' if SELFTEST else 'SUPPORTED'))
if len(dojo):
    for _,r in dojo.iterrows():
        if pd.notna(r.utility):rows.append(dict(benchmark='AgentDojo',model=r.model,family=r.family,slice=r.suite,metric='utility',score=float(r.utility),evidence_state='SELFTEST_ONLY' if SELFTEST else 'SUPPORTED'))
        if pd.notna(r.security):rows.append(dict(benchmark='AgentDojo',model=r.model,family=r.family,slice=r.suite,metric='security',score=float(r.security),evidence_state='SELFTEST_ONLY' if SELFTEST else 'SUPPORTED'))
if len(tau):
    for _,r in tau.iterrows():
        if pd.notna(r.reward):rows.append(dict(benchmark='tau3',model=r.model,family=r.family,slice=r.domain,metric='reward',score=float(r.reward),evidence_state='SELFTEST_ONLY' if SELFTEST else 'SUPPORTED'))
evidence=pd.DataFrame(rows);evidence.to_csv(RESULTS/'20_external_unified_evidence.csv',index=False)
summary=evidence.groupby(['benchmark','model','family','slice','metric','evidence_state'],dropna=False).score.agg(['count','mean','min','max']).reset_index().rename(columns={'count':'n_cases','mean':'mean_score','min':'min_score','max':'max_score'}) if len(evidence) else pd.DataFrame(columns=['benchmark','model','family','slice','metric','evidence_state','n_cases','mean_score','min_score','max_score']);summary.to_csv(RESULTS/'20_external_summary.csv',index=False)
def ci(vals,B):
    x=np.asarray(pd.Series(vals).dropna(),float)
    if len(x)<2:return (float(x.mean()) if len(x) else np.nan,np.nan,np.nan)
    rng=np.random.default_rng(SEED);m=np.array([rng.choice(x,len(x),replace=True).mean() for _ in range(B)]);return float(x.mean()),float(np.quantile(m,.025)),float(np.quantile(m,.975))
cirows=[]
for key,g in evidence.groupby(['benchmark','model','slice','metric']) if len(evidence) else []:
    a,b,c=ci(g.score,CFG['bootstrap']);cirows.append(dict(benchmark=key[0],model=key[1],slice=key[2],metric=key[3],n=len(g),mean=a,ci95_low=b,ci95_high=c))
cidf=pd.DataFrame(cirows);cidf.to_csv(RESULTS/'21_bootstrap_ci.csv',index=False);display(summary);display(cidf)

In [ ]:
# CELL 13 — STRICT CLAIM GATE / PAPER TABLES / FIGURES
def ok(df):return bool(len(df) and df.status.astype(str).str.startswith('SUPPORTED').any())
if SELFTEST: claims=pd.DataFrame([dict(claim=x,status='SELFTEST_ONLY') for x in ['BFCL-v4 external evidence','AgentDojo external evidence','tau3 external evidence','>=3 required external benchmarks','>=3 independent external model families']])
else:
    b,d,t=ok(bfcl_status),ok(dojo_status),ok(tau_status);f=set()
    for df,good in [(bfcl_status,b),(dojo_status,d),(tau_status,t)]:
        if good:f.update(df.loc[df.status.astype(str).str.startswith('SUPPORTED'),'family'].dropna().astype(str))
    claims=pd.DataFrame([dict(claim='BFCL-v4 external evidence',status='SUPPORTED' if b else 'MISSING'),dict(claim='AgentDojo external evidence',status='SUPPORTED' if d else 'MISSING'),dict(claim='tau3 external evidence',status='SUPPORTED' if t else 'MISSING'),dict(claim='>=3 required external benchmarks',status='SUPPORTED' if sum([b,d,t])>=3 else 'MISSING'),dict(claim='>=3 independent external model families',status='SUPPORTED' if len(f)>=3 else 'MISSING')])
claims.to_csv(RESULTS/'22_external_claim_checklist.csv',index=False);display(claims)
summary.to_latex(PAPER/'external_model_summary.tex',index=False,float_format='%.4f'); claims.to_latex(PAPER/'external_claim_checklist.tex',index=False); cidf.to_latex(PAPER/'external_bootstrap_ci.tex',index=False,float_format='%.4f')
for (b,m),g in summary.groupby(['benchmark','metric']) if len(summary) else []:
    gg=g.groupby('model',as_index=False).apply(lambda x:pd.Series({'score':np.average(x.mean_score,weights=np.maximum(x.n_cases,1))}),include_groups=False).reset_index(drop=True).sort_values('score');fig,ax=plt.subplots(figsize=(8,max(3,.55*len(gg)+1)));ax.barh(gg.model,gg.score);ax.set_title(f'{b}: {m}');ax.set_xlabel(m);fig.tight_layout();fig.savefig(PAPER/(re.sub(r'[^A-Za-z0-9]+','_',b)+'_'+re.sub(r'[^A-Za-z0-9]+','_',m)+'.png'),dpi=240,bbox_inches='tight');plt.close(fig)

In [ ]:
# CELL 14 — MANIFEST / FAILURE STATUS
frames=[]
for b,df in [('BFCL-v4',bfcl_status),('AgentDojo',dojo_status),('tau3',tau_status)]:
    x=df.copy();x['benchmark']=b;frames.append(x)
frames.append(optional.copy());master=pd.concat(frames,ignore_index=True,sort=False);master.to_csv(RESULTS/'23_master_run_status.csv',index=False)
manifest=dict(experiment='NTX-FINAL-EXTERNAL-COMPLETION',timestamp=datetime.now(timezone.utc).isoformat(),mode=MODE,selftest=SELFTEST,seed=SEED,models=WORKING,fixed_tau_user_simulator=FIXED_USER,claims=claims.to_dict('records'),config=CFG)
hashes={}
for n,root in [('results',RESULTS),('paper',PAPER),('environment',ENV),('logs',LOGS)]:
    for p in root.rglob('*'):
        if p.is_file() and p.name not in {'FINAL_MANIFEST.json','SHA256SUMS.txt'}:hashes[f'{n}/{p.relative_to(root)}']=sha(p)
manifest['artifact_sha256']=hashes;jd(RESULTS/'FINAL_MANIFEST.json',manifest);(RESULTS/'SHA256SUMS.txt').write_text('\n'.join(f'{h}  {n}' for n,h in sorted(hashes.items()))+'\n');display(master)

In [ ]:
# CELL 15 — DOWNLOAD EVERY DATA PACKAGE
rawstage=BASE/'_raw_export';shutil.rmtree(rawstage,ignore_errors=True);rawstage.mkdir();cp_tree(RAW,rawstage/'benchmark_raw');cp_tree(LOGS,rawstage/'logs');cp_tree(ENV,rawstage/'environment')
RAWZIP=ARCH/'NTX_EXTERNAL_RAW_DATA.zip';RESZIP=ARCH/'NTX_EXTERNAL_PROCESSED_RESULTS.zip';PAPERZIP=ARCH/'NTX_EXTERNAL_PAPER_ARTIFACTS.zip';zip_tree(rawstage,RAWZIP);zip_tree(RESULTS,RESZIP);zip_tree(PAPER,PAPERZIP)
masterstage=BASE/'_master_export';shutil.rmtree(masterstage,ignore_errors=True);masterstage.mkdir()
for p in [RAWZIP,RESZIP,PAPERZIP]:shutil.copy2(p,masterstage/p.name)
cp_tree(RESULTS,masterstage/'results');cp_tree(ENV,masterstage/'environment');cp_tree(LOGS,masterstage/'logs');cp_tree(PAPER,masterstage/'paper_artifacts')
MASTERZIP=ARCH/'NTX_FINAL_EXTERNAL_EXPERIMENTS_MASTER.zip';zip_tree(masterstage,MASTERZIP)
a=pd.DataFrame([dict(file=p.name,size_mib=round(p.stat().st_size/1024**2,3),sha256=sha(p)) for p in [RAWZIP,RESZIP,PAPERZIP,MASTERZIP]]);a.to_csv(ARCH/'ARCHIVE_MANIFEST.csv',index=False);display(a)
if DRIVE_BACKUP:
    d=Path(DRIVE_BACKUP);d.mkdir(parents=True,exist_ok=True)
    for p in [RAWZIP,RESZIP,PAPERZIP,MASTERZIP,ARCH/'ARCHIVE_MANIFEST.csv']:shutil.copy2(p,d/p.name)
    print('Drive backup',d)
status='SELFTEST_ONLY' if SELFTEST else ('SUPPORTED' if (claims.status=='SUPPORTED').all() else 'INCOMPLETE');jd(ARCH/'FINAL_COMPLETION_STATUS.json',dict(publication_external_gate=status,mode=MODE,selftest=SELFTEST,claims=claims.to_dict('records'),archives=a.to_dict('records')));print('FINAL GATE',status)
try:
    from google.colab import files
    for p in [RESZIP,PAPERZIP,RAWZIP,MASTERZIP]:files.download(str(p))
except Exception as e:print('Browser download unavailable:',repr(e),'Files remain in',ARCH)